<a href="https://colab.research.google.com/github/JSJeong-me/GPT-Web/blob/main/307_RAG_Multimodal_Colab_LowMemory_GitHubDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-modal RAG — Colab Low-Memory Edition

이 노트북은 원본 `RAG_Multimodal.ipynb`의 구조를 유지하되, 마지막 VLM 생성 단계를 일반 Colab에서도 실행하기 쉽도록 수정한 버전입니다.

- 기본 실행 경로: **Gemini API** 사용 → Qwen 7B/3B VLM을 로컬 GPU에 올리지 않음
- 선택 실행 경로: **Qwen/Qwen2-VL-2B-Instruct + 4-bit quantization** → Colab T4에서 가능한 저메모리 로컬 VLM 경로
- GitHub dataset 자동 다운로드: CornelliusYW/Multimodal-RAG-Implementation의 `dataset/`에서 PDF/MP3를 내려받음
- 이미지 검색: PDF 페이지 이미지를 CLIP 임베딩으로 검색
- 오디오 검색: Whisper 전사 결과를 SentenceTransformer 임베딩으로 검색
- 최종 답변: 검색된 이미지 + 오디오 전사 context + 사용자 질문을 VLM에 전달

In [ ]:
%%capture
!apt-get -qq update
!apt-get -qq install -y poppler-utils

# Colab 기본 torch는 그대로 사용합니다. torch를 강제로 재설치하지 않아 런타임 충돌 가능성을 줄입니다.
# Pillow 버전 충돌 문제를 해결하기 위해 <10 버전을 지정합니다.
!pip -q install -U \
  pdf2image "pillow==9.2.0" chromadb sentence-transformers transformers accelerate \
  librosa soundfile qwen-vl-utils google-genai bitsandbytes

In [ ]:
from pathlib import Path
import os
import gc
import json
import base64
import mimetypes
import shutil
import numpy as np
import torch
import chromadb
import librosa

from pdf2image import convert_from_path
from PIL import Image as PILImage
from IPython.display import display
from sentence_transformers import SentenceTransformer
from transformers import (
    CLIPProcessor,
    CLIPModel,
    WhisperProcessor,
    WhisperForConditionalGeneration,
)

# -----------------------------
# Runtime / experiment settings
# -----------------------------
DATASET_DIR = Path("dataset")
IMAGE_OUTPUT_DIR = Path("extracted_images")
CHROMA_DIR = Path("chroma_db")

DATASET_DIR.mkdir(exist_ok=True)
IMAGE_OUTPUT_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 기본은 Gemini API입니다. 로컬 GPU 메모리를 거의 쓰지 않습니다.
# 선택: "qwen_local" 로 바꾸면 Qwen/Qwen2-VL-2B-Instruct 4-bit 경로를 사용합니다.
VLM_BACKEND = "gemini"        # "gemini" or "qwen_local"

# Gemini 모델 후보입니다. 사용 중인 API 계정에서 앞 모델이 지원되지 않으면 다음 후보로 fallback합니다.
GEMINI_MODEL_CANDIDATES = [
    "gemini-3.5-flash",
    "gemini-2.5-flash",
    "gemini-2.0-flash",
]

# 로컬 Qwen 경로. 원본 7B 대신 2B 모델을 기본값으로 둡니다.
QWEN_LOCAL_MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"
QWEN_LOAD_IN_4BIT = True

# 원본 whisper-small 대신 whisper-tiny를 기본값으로 사용하여 Colab 메모리 사용량을 줄입니다.
WHISPER_MODEL_NAME = "openai/whisper-tiny"

# 검색 및 생성 설정
TOP_K = 2
MAX_NEW_TOKENS = 256
QUERY = "What are the healthiest ingredients to use in recipe you have?"

print("DEVICE:", DEVICE)
print("VLM_BACKEND:", VLM_BACKEND)
print("Dataset folder:", DATASET_DIR.resolve())

## 1. 데이터 준비: GitHub dataset 자동 다운로드 또는 수동 업로드

기본 실행은 아래 cell에서 GitHub repository의 `dataset/` 폴더를 읽어 PDF와 MP3 파일을 `dataset/` 로 자동 다운로드합니다.

Source URL:
`https://github.com/CornelliusYW/Multimodal-RAG-Implementation/tree/main/dataset`

- PDF: 페이지 이미지로 변환되어 이미지 검색 대상이 됩니다.
- MP3: Whisper로 전사되어 텍스트 검색 대상이 됩니다.
- GitHub API rate limit 문제가 있거나 다른 파일을 테스트하려면, 다음 수동 업로드 cell을 사용하세요.


In [ ]:
# -----------------------------
# 1A. GitHub dataset → local dataset/ 자동 다운로드
# -----------------------------
# 원본 예제 repository의 dataset 디렉터리에서 PDF와 MP3를 내려받습니다.
# Colab에서 이 cell을 먼저 실행하면 수동 업로드 없이 바로 예제를 실행할 수 있습니다.

# import requests
# from pathlib import Path
# import os # Added os import back

# GITHUB_DATASET_PAGE_URL = "https://github.com/CornelliusYW/Multimodal-RAG-Implementation/tree/main/dataset"
# GITHUB_DATASET_API_URL = "https://api.github.com/repos/CornelliusYW/Multimodal-RAG-Implementation/contents/dataset?ref=main"
# TARGET_SUFFIXES = {".pdf", ".mp3"}


# def _github_headers():
#     """Return GitHub API headers. Optional GITHUB_TOKEN helps avoid low rate limits."""
#     headers = {"Accept": "application/vnd.github+json"}

#     token = os.getenv("GITHUB_TOKEN") # Re-enabled GITHUB_TOKEN usage

#     # Optional Colab Secret fallback: store a GitHub token as GITHUB_TOKEN if needed.
#     if not token:
#         try:
#             from google.colab import userdata
#             token = userdata.get("GITHUB_TOKEN")
#         except Exception:
#             token = None

#     if token:
#         headers["Authorization"] = f"Bearer {token}" # Re-enabled Authorization header

#     return headers


# def download_github_dataset(
#     dataset_dir: Path = DATASET_DIR,
#     api_url: str = GITHUB_DATASET_API_URL,
#     suffixes = TARGET_SUFFIXES,
#     overwrite: bool = False,
# ):
#     """Download PDF/MP3 files from a GitHub repository folder into dataset/."""
#     dataset_dir = Path(dataset_dir)
#     dataset_dir.mkdir(parents=True, exist_ok=True)

#     headers = _github_headers()
#     print("GitHub dataset source:", GITHUB_DATASET_PAGE_URL)
#     print("Fetching directory metadata...")

#     response = requests.get(api_url, headers=headers, timeout=30)
#     if response.status_code != 200:
#         raise RuntimeError(
#             f"GitHub API request failed: HTTP {response.status_code}\n"
#             f"URL: {api_url}\n"
#             f"Response: {response.text[:500]}"
#         )

#     items = response.json()
#     if not isinstance(items, list):
#         raise RuntimeError(f"Unexpected GitHub API response: {items}")

#     files = []
#     for item in items:
#         if item.get("type") != "file":
#             continue
#         name = item.get("name", "")
#         suffix = Path(name).suffix.lower()
#         if suffix in suffixes:
#             files.append(item)

#     if not files:
#         raise RuntimeError(
#             f"No target files found in GitHub dataset folder. "
#             f"Expected suffixes: {sorted(suffixes)}"
#         )

#     downloaded = []
#     skipped = []

#     for item in files:
#         name = item["name"]
#         download_url = item.get("download_url")
#         if not download_url:
#             print(f"Skip {name}: download_url is missing")
#             continue

#         target_path = dataset_dir / name
#         if target_path.exists() and not overwrite:
#             skipped.append(target_path)
#             continue

#         print(f"Downloading {name} → {target_path}")
#         with requests.get(download_url, headers=headers, stream=True, timeout=120) as r:
#             r.raise_for_status()
#             with open(target_path, "wb") as f:
#                 for chunk in r.iter_content(chunk_size=1024 * 1024):
#                     if chunk:
#                         f.write(chunk)

#         downloaded.append(target_path)

#     print("\nDownload summary")
#     print(" downloaded:", len(downloaded))
#     print(" skipped existing:", len(skipped))
#     print(" dataset files:")
#     for p in sorted(dataset_dir.glob("*")):
#         if p.suffix.lower() in suffixes:
#             print(f"  - {p.name} ({p.stat().st_size / 1024:.1f} KB)")

#     return downloaded, skipped


# # 기본 실행: repository dataset의 PDF/MP3를 dataset/ 폴더에 준비합니다.
# # 이미 파일이 있으면 overwrite=False 때문에 다시 받지 않습니다.
# downloaded_files, skipped_files = download_github_dataset(overwrite=False)

In [ ]:
# 선택 실행: GitHub 자동 다운로드 대신 로컬 PC/Colab 파일 업로드 UI를 사용하려면 실행합니다.
# 이미 GitHub 다운로드 cell을 실행했거나 dataset 폴더에 파일을 넣었다면 이 cell은 실행하지 않아도 됩니다.

def upload_files_to_dataset():
    try:
        from google.colab import files
    except Exception:
        print("google.colab 환경이 아닙니다. 파일을 직접 dataset/ 폴더에 넣어 주세요.")
        return

    uploaded = files.upload()
    for name, data in uploaded.items():
        target = DATASET_DIR / Path(name).name
        with open(target, "wb") as f:
            f.write(data)
        print("saved:", target)

print("Current dataset files:")
for p in sorted(DATASET_DIR.glob("*")):
    print(" -", p.name)

# 필요할 때만 아래 주석을 해제하세요.
# upload_files_to_dataset()

In [ ]:
# -----------------------------
# 2. PDF → page images
# -----------------------------
def convert_pdfs_to_images(folder: Path, image_output_dir: Path, dpi: int = 100):
    image_output_dir.mkdir(exist_ok=True)

    pdf_files = sorted([p for p in folder.glob("*.pdf")])
    all_images = {}

    if not pdf_files:
        print(f"No PDF files found in {folder}. Image retrieval will be empty.")
        return all_images

    for doc_id, pdf_path in enumerate(pdf_files):
        print(f"Converting PDF: {pdf_path.name}")
        # Request PNG format directly to avoid PpmImageFile issues
        pages = convert_from_path(str(pdf_path), dpi=dpi, fmt="png")

        image_paths = []
        for i, page in enumerate(pages):
            image_path = image_output_dir / f"{doc_id}_page_{i}.png"
            # Explicitly convert to RGB to avoid AttributeError with PpmImageFile mode setter
            page.convert("RGB").save(image_path, "PNG")
            image_paths.append(str(image_path))

        all_images[doc_id] = image_paths

    return all_images

all_images = convert_pdfs_to_images(DATASET_DIR, IMAGE_OUTPUT_DIR)
print("PDF image pages:", sum(len(v) for v in all_images.values()))

In [ ]:
import numpy as np
import torch
from PIL import Image as PILImage

# -----------------------------
# 3. CLIP image/text embeddings
# -----------------------------
# 원본 notebook의 get_image_features(...).pooler_output 접근은 transformers 버전에 따라 오류가 납니다.
# 여기서는 get_image_features() / get_text_features()의 tensor를 직접 사용하고 L2-normalize합니다.

CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"

clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)

def _normalize(features: torch.Tensor) -> torch.Tensor:
    return features / features.norm(dim=-1, keepdim=True).clamp(min=1e-12)

@torch.no_grad()
def embed_image_for_search(image_path: str) -> np.ndarray:
    image = PILImage.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt").to(DEVICE)
    features_output = clip_model.get_image_features(**inputs)
    features = features_output.pooler_output if hasattr(features_output, 'pooler_output') else features_output
    print(f"DEBUG: Features type before norm: {type(features)}")
    print(f"DEBUG: Features shape before norm: {features.shape}")
    print(f"DEBUG: Features has NaN before norm: {torch.isnan(features).any()}")
    features = _normalize(features)
    print(f"DEBUG: Features type after norm: {type(features)}")
    print(f"DEBUG: Features shape after norm: {features.shape}")
    print(f"DEBUG: Features has NaN after norm: {torch.isnan(features).any()}")
    return features.squeeze(0).detach().cpu().numpy().astype("float32")

@torch.no_grad()
def embed_text_for_image_search(text: str) -> np.ndarray:
    inputs = clip_processor(text=[text], return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    features_output = clip_model.get_text_features(**inputs)
    features = features_output.pooler_output if hasattr(features_output, 'pooler_output') else features_output
    features = _normalize(features)
    return features.squeeze(0).detach().cpu().numpy().astype("float32")

image_embeddings = {}
for doc_id, paths in all_images.items():
    image_embeddings[doc_id] = [embed_image_for_search(p) for p in paths]

print("Embedded image pages:", sum(len(v) for v in image_embeddings.values()))


In [ ]:
# -----------------------------
# 4. Audio → Whisper transcription
# -----------------------------
whisper_processor = WhisperProcessor.from_pretrained(WHISPER_MODEL_NAME)
whisper_model = WhisperForConditionalGeneration.from_pretrained(WHISPER_MODEL_NAME).to(DEVICE).eval()

@torch.no_grad()
def transcribe_audio(audio_path: str, chunk_length: int = 30):
    audio, sr = librosa.load(audio_path, sr=16000)
    chunk_size = chunk_length * sr
    chunks = [audio[i:i + chunk_size] for i in range(0, len(audio), chunk_size)]

    transcription_chunks = []
    for idx, chunk in enumerate(chunks):
        if len(chunk) == 0:
            continue
        inputs = whisper_processor(chunk, sampling_rate=sr, return_tensors="pt")
        input_features = inputs.input_features.to(DEVICE)

        predicted_ids = whisper_model.generate(
            input_features=input_features,
            max_new_tokens=128,
        )
        text = whisper_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()
        if text:
            transcription_chunks.append(text)
        print(f"  chunk {idx+1}/{len(chunks)}:", text[:80])

    full_transcription = " ".join(transcription_chunks)
    return full_transcription, transcription_chunks

audio_files = sorted([p for p in DATASET_DIR.glob("*.mp3")])
audio_transcriptions = {}

if not audio_files:
    print(f"No MP3 files found in {DATASET_DIR}. Audio context retrieval will be empty.")
else:
    for audio_id, audio_path in enumerate(audio_files):
        print("Transcribing:", audio_path.name)
        full_transcription, chunks = transcribe_audio(str(audio_path))
        audio_transcriptions[audio_id] = {
            "audio_path": str(audio_path),
            "full_transcription": full_transcription,
            "chunks": chunks,
        }

print("Audio files:", len(audio_files))
print("Audio chunks:", sum(len(v["chunks"]) for v in audio_transcriptions.values()))

In [ ]:
# -----------------------------
# 5. Build Chroma vector DB
# -----------------------------
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
text_embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)

# Recreate collections for a clean run.
for name in ["image_collection", "audio_collection"]:
    try:
        client.delete_collection(name=name)
    except Exception:
        pass

image_collection = client.create_collection(
    name="image_collection",
    metadata={"hnsw:space": "cosine"},
)
audio_collection = client.create_collection(
    name="audio_collection",
    metadata={"hnsw:space": "cosine"},
)

# Add image embeddings.
for doc_id, embeddings in image_embeddings.items():
    for i, embedding in enumerate(embeddings):
        image_collection.add(
            ids=[f"image_{doc_id}_{i}"],
            embeddings=[embedding.flatten().tolist()],
            metadatas=[{
                "doc_id": str(doc_id),
                "page_index": str(i),
                "image_path": all_images[doc_id][i],
            }],
        )

# Add audio transcript chunk embeddings.
for audio_id, transcription_data in audio_transcriptions.items():
    for chunk_id, chunk in enumerate(transcription_data["chunks"]):
        if not chunk.strip():
            continue
        chunk_embedding = text_embedding_model.encode(chunk, normalize_embeddings=True)
        audio_collection.add(
            ids=[f"audio_{audio_id}_chunk_{chunk_id}"],
            embeddings=[chunk_embedding.tolist()],
            metadatas=[{
                "audio_id": str(audio_id),
                "audio_path": transcription_data["audio_path"],
                "chunk_id": str(chunk_id),
            }],
            documents=[chunk],
        )

print("image_collection count:", image_collection.count())
print("audio_collection count:", audio_collection.count())

In [ ]:
# -----------------------------
# 6. Retrieve relevant images + audio chunks
# -----------------------------
def retrieve_data(query: str, top_k: int = 2):
    retrieved_images = []
    retrieved_chunks = []

    if image_collection.count() > 0:
        k_img = min(top_k, image_collection.count())
        query_embedding_image = embed_text_for_image_search(query)
        image_results = image_collection.query(
            query_embeddings=[query_embedding_image.tolist()],
            n_results=k_img,
        )
        retrieved_images = [
            m["image_path"]
            for m in image_results.get("metadatas", [[]])[0]
            if m and "image_path" in m
        ]

    if audio_collection.count() > 0:
        k_audio = min(top_k, audio_collection.count())
        query_embedding_audio = text_embedding_model.encode(query, normalize_embeddings=True)
        audio_results = audio_collection.query(
            query_embeddings=[query_embedding_audio.tolist()],
            n_results=k_audio,
        )
        retrieved_chunks = audio_results.get("documents", [[]])[0] or []

    return retrieved_images, retrieved_chunks

retrieved_images, retrieved_chunks = retrieve_data(QUERY, TOP_K)

print("QUERY:", QUERY)
print("\nRetrieved Images:")
for p in retrieved_images:
    print(" -", p)

print("\nRetrieved Audio/Text Chunks:")
for c in retrieved_chunks:
    print(" -", c[:300])

In [ ]:
# -----------------------------
# 7. Display retrieved images
# -----------------------------
def display_retrieved_images(image_paths):
    if not image_paths:
        print("No retrieved images.")
        return

    for image_path in image_paths:
        try:
            img = PILImage.open(image_path).convert("RGB")
            print(f"Displaying image: {image_path}")
            display(img)
        except Exception as e:
            print(f"Error displaying image {image_path}: {e}")

display_retrieved_images(retrieved_images)

## 8. VLM 생성 Backend A — Gemini API, 기본 추천

이 경로는 `Qwen/Qwen2-VL-7B-Instruct` 또는 3B 모델을 Colab GPU에 올리지 않습니다. 따라서 일반 Colab에서 가장 안정적입니다.

Colab에서 먼저 다음 중 하나를 설정하세요.

```python
import os
os.environ["GEMINI_API_KEY"] = "YOUR_API_KEY"
```

또는 Colab Secret에 `GEMINI_API_KEY`를 저장해 두면 아래 코드가 자동으로 읽습니다.

In [ ]:
# -----------------------------
# 8A. Gemini generation backend
# -----------------------------
def _load_gemini_api_key():
    if os.getenv("GEMINI_API_KEY"):
        return os.getenv("GEMINI_API_KEY")

    # Colab Secret fallback
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key:
            os.environ["GEMINI_API_KEY"] = key
            return key
    except Exception:
        pass

    return None

def _read_image_as_gemini_part(image_path: str):
    mime_type = mimetypes.guess_type(image_path)[0] or "image/png"
    with open(image_path, "rb") as f:
        image_b64 = base64.b64encode(f.read()).decode("utf-8")
    return {
        "type": "image",
        "data": image_b64,
        "mime_type": mime_type,
    }

def build_grounded_prompt(query: str, retrieved_chunks, max_context_chars: int = 4000):
    context = "\n".join([f"- {c}" for c in retrieved_chunks if c and c.strip()])
    context = context[:max_context_chars] if context else "(No retrieved audio/text context.)"

    return f"""
You are a multimodal RAG assistant.

Use the retrieved images and retrieved audio/text context to answer the user's question.
If the retrieved context is insufficient, say so clearly instead of guessing.

User question:
{query}

Retrieved audio/text context:
{context}

Answer:
""".strip()

def generate_answer_gemini(
    query: str,
    retrieved_images,
    retrieved_chunks,
    model_candidates=None,
):
    from google import genai

    api_key = _load_gemini_api_key()
    if not api_key:
        raise RuntimeError(
            "GEMINI_API_KEY is not set. In Colab, run: "
            "import os; os.environ['GEMINI_API_KEY']='YOUR_API_KEY'"
        )

    model_candidates = model_candidates or GEMINI_MODEL_CANDIDATES
    client = genai.Client(api_key=api_key)

    prompt = build_grounded_prompt(query, retrieved_chunks)
    input_parts = [{"type": "text", "text": prompt}]

    for image_path in retrieved_images[:TOP_K]:
        input_parts.append(_read_image_as_gemini_part(image_path))

    errors = []

    for model_name in model_candidates:
        try:
            # Current Google GenAI SDK path.
            if hasattr(client, "interactions"):
                interaction = client.interactions.create(
                    model=model_name,
                    input=input_parts,
                )
                return interaction.output_text

            # Older google-genai fallback path.
            image_objects = [PILImage.open(p).convert("RGB") for p in retrieved_images[:TOP_K]]
            response = client.models.generate_content(
                model=model_name,
                contents=[prompt] + image_objects,
            )
            return getattr(response, "text", str(response))

        except Exception as e:
            errors.append(f"{model_name}: {type(e).__name__}: {e}")

    raise RuntimeError("Gemini generation failed for all candidate models:\n" + "\n".join(errors))

## 9. VLM 생성 Backend B — Qwen Local 저메모리 선택 경로

Gemini API를 쓰지 않고 로컬 VLM을 실행해야 할 때만 사용합니다.

원본의 `Qwen/Qwen2-VL-7B-Instruct` 대신 기본값을 `Qwen/Qwen2-VL-2B-Instruct`로 낮추고, 4-bit quantization을 사용합니다. 그래도 Gemini API 경로보다 GPU 메모리를 많이 사용합니다.

In [ ]:
# -----------------------------
# 8B. Optional Qwen local backend
# -----------------------------
def generate_answer_qwen_local(
    query: str,
    retrieved_images,
    retrieved_chunks,
    model_name: str = QWEN_LOCAL_MODEL_NAME,
    load_in_4bit: bool = QWEN_LOAD_IN_4BIT,
):
    if not torch.cuda.is_available():
        print("CUDA GPU가 없습니다. CPU 실행은 매우 느릴 수 있습니다.")

    try:
        from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
        from qwen_vl_utils import process_vision_info
    except Exception as e:
        raise RuntimeError(
            "Qwen local backend requires recent transformers and qwen-vl-utils. "
            "Install/upgrade them and restart runtime."
        ) from e

    quantization_config = None
    if load_in_4bit and torch.cuda.is_available():
        from transformers import BitsAndBytesConfig
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

    model_kwargs = {
        "device_map": "auto",
        "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
    }
    if quantization_config is not None:
        model_kwargs["quantization_config"] = quantization_config

    print("Loading local Qwen model:", model_name)
    qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(
        model_name,
        **model_kwargs,
    ).eval()

    # 메모리 절감을 위해 이미지 해상도 상한을 낮춥니다.
    qwen_processor = AutoProcessor.from_pretrained(
        model_name,
        min_pixels=128 * 128,
        max_pixels=512 * 512,
    )

    prompt = build_grounded_prompt(query, retrieved_chunks)

    content = []
    for image_path in retrieved_images[:TOP_K]:
        content.append({"type": "image", "image": image_path})
    content.append({"type": "text", "text": prompt})

    messages = [{"role": "user", "content": content}]

    text = qwen_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)
    inputs = qwen_processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    target_device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = inputs.to(target_device)

    with torch.no_grad():
        generated_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = qwen_processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    # 메모리 회수
    del qwen_model, qwen_processor, inputs, generated_ids
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return output_text

In [ ]:
# -----------------------------
# 10. Generate final answer
# -----------------------------
if VLM_BACKEND == "gemini":
    final_answer = generate_answer_gemini(QUERY, retrieved_images, retrieved_chunks)
elif VLM_BACKEND == "qwen_local":
    final_answer = generate_answer_qwen_local(QUERY, retrieved_images, retrieved_chunks)
else:
    raise ValueError("VLM_BACKEND must be either 'gemini' or 'qwen_local'.")

print(final_answer)

## 변경 요약

원본 notebook 대비 주요 변경 사항은 다음과 같습니다.

0. GitHub repository의 `dataset/` 자동 다운로드 cell을 추가하여 PDF/MP3 수동 업로드 없이 예제를 실행할 수 있게 했습니다.

1. 최종 VLM 기본값을 로컬 Qwen 7B/3B에서 **Gemini API**로 변경하여 Colab GPU 메모리 부담을 줄였습니다.
2. 로컬 실행이 필요한 경우를 위해 **Qwen/Qwen2-VL-2B-Instruct + 4-bit quantization** 선택 경로를 추가했습니다.
3. CLIP 이미지 임베딩 코드에서 `.pooler_output` 접근을 제거하고, `get_image_features()` / `get_text_features()` 기반으로 수정했습니다.
4. `whisper-small` 대신 `whisper-tiny`를 기본값으로 사용하여 오디오 전사 메모리를 줄였습니다.
5. PDF나 MP3가 없는 경우에도 notebook이 즉시 중단되지 않도록 방어 로직을 추가했습니다.
6. ChromaDB collection 생성, 검색 top-k, 이미지 표시, Gemini API key 로딩을 Colab 친화적으로 정리했습니다.

In [ ]:
import os

print(os.listdir('dataset'))